# Klingon Translator - Exploration Notebook

Local notebook for experimenting with the Klingon translation pipeline.

**Prerequisites:**
- `conda activate klingon && pip install -e .`
- Run `python -m klingon_translator.data.download` to get training data
- Fine-tune on Colab first (see `colab/klingon_training.ipynb`)

In [ ]:
# Verify imports
import klingon_translator
print(f"Package version: {klingon_translator.__version__}")

## 1. Explore the Data

In [ ]:
import json
from klingon_translator.utils.config import PROCESSED_DATA_DIR

def load_jsonl(path):
    with open(path, encoding="utf-8") as f:
        return [json.loads(line) for line in f if line.strip()]

train_file = PROCESSED_DATA_DIR / "train.jsonl"
if train_file.exists():
    train_pairs = load_jsonl(PROCESSED_DATA_DIR / "train.jsonl")
    val_pairs = load_jsonl(PROCESSED_DATA_DIR / "val.jsonl")
    test_pairs = load_jsonl(PROCESSED_DATA_DIR / "test.jsonl")
    print(f"Train: {len(train_pairs):,} pairs")
    print(f"Val:   {len(val_pairs):,} pairs")
    print(f"Test:  {len(test_pairs):,} pairs")
    print()
    for p in train_pairs[:5]:
        print(f"  EN:  {p['en']}")
        print(f"  TLH: {p['tlh']}")
        print()
else:
    print("No data yet. Run: python -m klingon_translator.data.download")

## 2. Inspect the Klingon Tokenizer

In [ ]:
from klingon_translator.model.tokenizer import collect_klingon_text, train_klingon_spm
from klingon_translator.utils.config import MODELS_DIR
import sentencepiece as spm
from pathlib import Path

# Check if SPM model already exists
spm_path = MODELS_DIR / "klingon_spm" / "klingon_spm.model"
if not spm_path.exists():
    print("Training SentencePiece model...")
    text = collect_klingon_text()
    spm_path = train_klingon_spm(text)
else:
    print(f"Using existing SPM model: {spm_path}")

sp = spm.SentencePieceProcessor()
sp.load(str(spm_path))
print(f"Vocab size: {sp.get_piece_size()}")

# Test tokenization on various Klingon phrases
test_phrases = [
    "nuqneH",
    "Qapla'",
    "tlhIngan maH!",
    "Heghlu'meH QaQ jajvam.",
    "bortaS bIr jablu'DI' reH QaQqu' nay'.",
]
print()
for phrase in test_phrases:
    pieces = sp.encode(phrase, out_type=str)
    print(f"  {phrase}")
    print(f"    -> {pieces}")
    print()

## 3. Load the Translator

After fine-tuning on Colab, download `models/fine-tuned/` from Google Drive.
The translator will auto-detect available models.

In [ ]:
from klingon_translator.model.translator import KlingonTranslator, EXTENDED_MODEL_DIR
from klingon_translator.utils.config import MODELS_DIR

fine_tuned_dir = MODELS_DIR / "fine-tuned"

if fine_tuned_dir.exists():
    print("Loading fine-tuned model...")
    translator = KlingonTranslator(fine_tuned_dir)
elif EXTENDED_MODEL_DIR.exists():
    print("Loading extended (pre-training) model...")
    translator = KlingonTranslator()
else:
    print("No fine-tuned or extended model found.")
    print("Options:")
    print("  1. Fine-tune on Colab: colab/klingon_training.ipynb")
    print("  2. Run tokenizer extension: python -m klingon_translator.model.tokenizer")
    print()
    print("Loading base NLLB model (no Klingon support)...")
    translator = KlingonTranslator()

## 4. Translate!

In [ ]:
# English -> Klingon
print("=== English -> Klingon ===")
en_phrases = [
    "Today is a good day to die.",
    "Success!",
    "What do you want?",
    "We are Klingons!",
    "Where is the bathroom?",
    "I don't understand.",
    "Thank you.",
]

if translator.has_klingon:
    for phrase in en_phrases:
        result = translator.to_klingon(phrase)
        print(f"  {phrase}")
        print(f"    -> {result}")
        print()
else:
    print("  (Klingon not supported - fine-tune first)")
    print("  Testing English -> French instead:")
    for phrase in en_phrases[:3]:
        result = translator.translate(phrase, tgt_lang="fra_Latn")
        print(f"  {phrase} -> {result}")

In [ ]:
# Klingon -> English
print("=== Klingon -> English ===")
tlh_phrases = [
    "Qapla'!",
    "nuqneH.",
    "tlhIngan maH!",
    "HIja'.",
    "ghobe'.",
    "yIDoghQo'!",
    "Heghlu'meH QaQ jajvam.",
]

if translator.has_klingon:
    for phrase in tlh_phrases:
        result = translator.to_english(phrase)
        print(f"  {phrase}")
        print(f"    -> {result}")
        print()
else:
    print("  (Klingon not supported - fine-tune first)")

## 5. Evaluate on Test Set

In [ ]:
# Run BLEU evaluation on test data (requires fine-tuned model)
if translator.has_klingon and 'test_pairs' in dir():
    import sacrebleu

    # Sample evaluation (full test set can be slow on CPU)
    sample_size = min(100, len(test_pairs))
    sample = test_pairs[:sample_size]

    print(f"Evaluating on {sample_size} test pairs (CPU - may be slow)...")

    # en -> tlh
    en_texts = [p["en"] for p in sample]
    tlh_refs = [p["tlh"] for p in sample]
    tlh_preds = translator.translate_batch(en_texts, num_beams=3)
    bleu_en2tlh = sacrebleu.corpus_bleu(tlh_preds, [tlh_refs])
    print(f"  en->tlh BLEU: {bleu_en2tlh.score:.2f}")

    # tlh -> en
    tlh_texts = [p["tlh"] for p in sample]
    en_refs = [p["en"] for p in sample]
    en_preds = translator.translate_batch(
        tlh_texts,
        src_lang="tlh_Latn", tgt_lang="eng_Latn",
        num_beams=3,
    )
    bleu_tlh2en = sacrebleu.corpus_bleu(en_preds, [en_refs])
    print(f"  tlh->en BLEU: {bleu_tlh2en.score:.2f}")
else:
    print("Skipping evaluation (need fine-tuned model and test data)")

## 6. Interactive Translation

In [ ]:
# Interactive cell - edit the text and re-run!
text_to_translate = "Hello, my friend!"
direction = "en_to_tlh"  # or "tlh_to_en"

if translator.has_klingon:
    if direction == "en_to_tlh":
        result = translator.to_klingon(text_to_translate)
    else:
        result = translator.to_english(text_to_translate)
    print(f"Input:  {text_to_translate}")
    print(f"Output: {result}")
else:
    print("Fine-tune the model first!")